In [1]:
!pip install spacy
!python -m spacy download en_core_web_sm
import math
import torch
import random
import numpy as np
import spacy
import pandas as pd
from torch import nn, optim

from datetime import datetime
from dataclasses import dataclass, asdict
from tokenizer import TinyStoriesTokenizer
from self_attention import MultiHeadSelfAttention


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 213.0 MB/s  0:00:00

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


### Transformer Block Architecture
This section implements the core computational unit of the model. It consists of a Multi-Head Self-Attention mechanism followed by a Position-wise Feed-Forward Network (FFN).

In [2]:
class PositionwiseFFN(nn.Module):
    """
    The position-wise FFN that follows after the self-attention computation.
    Vectors are projected to 4x the dimensionality and then projected down
    again after relu application.
    """

    def __init__(self, vector_dim, dropout_prob) :
        super().__init__()
        self.fc1 = nn.Linear(vector_dim, 4*vector_dim, bias=True)
        self.fc2 = nn.Linear(4*vector_dim, vector_dim, bias=True)
        self.dropout = nn.Dropout(dropout_prob)

    def forward(self, x):
        return self.fc2(self.dropout(torch.relu(self.fc1(x))))

class Block(nn.Module):
    """
    Transformer encoder block.

    This version differs from the original version in  [Vaswani et al. NeurIPS 2017],
    and applies the LayerNorm before the self-attention, and before the FFN, as this
    has proved to be beneficial (see [Nguyen and Salazar 2019]).
    """

    def __init__(self, vector_dim, n_heads, block_size, dropout_prob):
        super().__init__()
        att_dim = vector_dim // n_heads
        self.attn = MultiHeadSelfAttention(vector_dim, n_heads, block_size, is_causal=True)
        self.ffn = PositionwiseFFN(vector_dim, dropout_prob)
        self.dropout = nn.Dropout(dropout_prob)
        self.ln1 = nn.LayerNorm(vector_dim)
        self.ln2 = nn.LayerNorm(vector_dim)

    def forward(self, x):
        x1 = self.ln1(x)
        x2 = x + self.dropout(self.attn(x1))
        x3 = self.ln2(x2)
        x4 = x2 + self.dropout(self.ffn(x3))
        return x4

### Model Architecture

This section defines the internal structure of the TinyStoriesLM. It serves as the core engine to process text and extract the internal vector representations needed for linguistic analysis.

In [3]:
# ============= Hyper-parameters for training ============== #
@dataclass
class Config :
    vocab_size: int = 5000  # This number should agree with the tokenizer
    number_of_transformer_blocks: int = 4
    number_of_attention_heads: int = 4
    vector_dim: int = 256
    block_size: int = 512
    dropout_prob: float = 0.1
    batch_size: int = 8
    learning_rate: float = 0.0005
    weight_decay: float = 0.000001
    no_of_epochs: int = 1


class TinyStoriesLM(nn.Module):

    def __init__(self, config):
        super(TinyStoriesLM, self).__init__()
        self.config = config
        self.embed =  nn.Embedding(config.vocab_size, config.vector_dim)
        self.positional = nn.Parameter(torch.randn(1, config.block_size, config.vector_dim))
        modules = [Block(config.vector_dim,\
                         config.number_of_attention_heads,\
                         config.block_size,\
                         config.dropout_prob) for _ in range(config.number_of_transformer_blocks)]
        self.transformers = nn.ModuleList(modules)
        self.final = nn.Linear(config.vector_dim, config.vocab_size)

    def forward(self, x):
        # x size (B, S)
        # each element is a token-word -> transf it to vector 
        B, S = x.shape
        
        # Embedding (B, S, vector_dim)
        token_embeddings = self.embed(x) # each num.token is now a 256 len vector

        # Position Embedding - size ( 1, blocksize, vector_dim) -> S increasing as model generates more words
        pos_embeddings = self.positional[:, :S, :] # (1, S, vector_dim)

        x = token_embeddings + pos_embeddings # (B, S, vector_dim)
        
        # TRANSFORMERS BLOCK
        for block in self.transformers:
            x = block(x)
        # output - same len but with more context

        # Project to 5_000 words. For each pos - how probable for next word to be ... from 0-4999
        logits = self.final(x) # (B, S, vector_dim) to (B, S, vocab_size)
        
        # YOUR CODE HERE
        return logits
        
    @classmethod
    def load(cls, checkpoint_path, device='cpu'):
        """
        Loads a model from a checkpoint file.
        Automatically reconstructs the config and model architecture.
        """
        checkpoint = torch.load(checkpoint_path, map_location=device)
        config = Config(**checkpoint['config'])
        model = cls(config)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.to(device)
    
        print(f"Model loaded from {checkpoint_path} (Epoch {checkpoint['epoch']}, iteration {checkpoint['iteration']})")
        return model


### Dataset Management and Data Loading
This section provides the infrastructure to efficiently handle the TinyStories dataset. It focuses on loading tokenized data for model inference and subsequent linguistic analysis.

In [4]:
from torch.utils.data import Sampler, SubsetRandomSampler
from torch.utils.data import Dataset, DataLoader
from tokenizer import TinyStoriesTokenizer

class TinyStoriesDataset(Dataset):
    def __init__(self, data_file, block_size):
        """
        data_file: path to the .bin file (uint16 array of token IDs)
        block_size: the context window (e.g., 256 or 512 tokens)
        """

        # Memory-map the data file (RAM usage stays near zero!)
        self.data = np.memmap(data_file, dtype=np.uint16, mode='r')
        self.block_size = block_size

    def __len__(self):
        # We subtract block_size to ensure we don't go out of bounds
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # Pull a chunk of length block_size + 1 (data and target)
        chunk = self.data[idx : idx + self.block_size + 1]

        # Convert to torch tensors
        x = torch.from_numpy(chunk[:-1].astype(np.int64)) # Input
        y = torch.from_numpy(chunk[1:].astype(np.int64))  # Target (shifted by 1)

        return x, y 


# This class facilitates re-starting training without repeating any past training examples
class ResumableRandomSampler(Sampler):
    def __init__(self, data_source, start_index=0, seed=42):
        self.data_source = data_source
        self.start_index = start_index
        self.seed = seed
        self.generator = torch.Generator()
        self.generator.manual_seed(self.seed)
        
        # 1. Generate the full shuffled list of indices once
        self.indices = torch.randperm(len(self.data_source), generator=self.generator).tolist()

    def __iter__(self):
        # 2. Slice the indices to only include those from start_index onwards
        return iter(self.indices[self.start_index:])

    def __len__(self):
        return len(self.indices) - self.start_index

### POS TAGGER WITH SPACY
This function gets a text and returns the POS tag of each word in a tuple

In [5]:
def get_POS_tags(text):
    doc = nlp(text)
    tags_list = []
    
    for token in doc:
        tags_list.append((token.text, token.pos_))
    return tags_list

In [6]:
sample_idx = 42  
training_dataset = TinyStoriesDataset('/datasets/dd2417/train.bin', Config.block_size)
tokenizer = TinyStoriesTokenizer.load('/datasets/dd2417/tokenizer.json')
x_sample, _ = training_dataset[sample_idx]
tokens_x = x_sample[:15].tolist()

nlp = spacy.load("en_core_web_sm")
full_text = "".join(tokenizer.vocab[idx] for idx in tokens_x)
print(f"Text: {full_text}\n")

#Get tags
pos_results = get_POS_tags(full_text)


#Table
print(f"{'Token (ID)':<15} | {'WORD':<10} | {'POS TAG':<15}")
print("-" * 50)



for token_id in tokens_x:
    word_fragment = tokenizer.vocab[token_id]
    pos_tag = "" 
    for word, tag in pos_results:
        if word_fragment.strip() in word and word_fragment.strip() != "":
            pos_tag = tag
            break
            
    print(f"{str(token_id):<15} | {word_fragment:<10} | {pos_tag:<15}")

Text: , but in his haste, he had forgot to bring it with him

Token (ID)      | WORD       | POS TAG        
--------------------------------------------------
70              | ,          | PUNCT          
212             |  but       | CCONJ          
135             |  in        | ADP            
158             |  his       | PRON           
114             |  ha        | NOUN           
1401            | ste        | NOUN           
70              | ,          | PUNCT          
96              |  he        | PRON           
195             |  had       | AUX            
1204            |  forgot    | VERB           
81              |  to        | PART           
1572            |  bring     | VERB           
126             |  it        | PRON           
147             |  with      | ADP            
264             |  him       | PRON           


In [ ]:
import os

# ======================= Training ======================= #

# Remove the checkpoint file if you want to train from scratch
checkpoint_path = 'last_checkpoint_next_word.pt'
RESUME_TRAINING = os.path.exists(checkpoint_path) 

seed = 42
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print( "Running on", device )

tokenizer = TinyStoriesTokenizer.load('/datasets/dd2417/tokenizer.json')

if RESUME_TRAINING:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    print(f"Model loaded from {checkpoint_path} (Epoch {checkpoint['epoch']+1}, iteration {checkpoint['iteration']})")
    config = Config(**checkpoint['config'])
    lm = TinyStoriesLM(config)
    # Sanity check
    if tokenizer.vocab_size != lm.config.vocab_size:
        print("WARNING: The tokenizer's and the model's vocab_size are different!")
    lm.load_state_dict(checkpoint['model_state_dict'])
    lm.to(device) 
else :
    config = Config()
    config.vocab_size = tokenizer.vocab_size
    lm = TinyStoriesLM(config).to(device)

lm_optimizer = torch.optim.AdamW(lm.parameters(), lr=lm.config.learning_rate, weight_decay=config.weight_decay)

if RESUME_TRAINING:
    lm_optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

training_dataset = TinyStoriesDataset('/datasets/dd2417/train.bin', config.block_size)

# Development set 
dev_dataset = TinyStoriesDataset('/datasets/dd2417/dev.bin', config.block_size)
# Create indices that jump by the block size. This ensures we only see each 
# chunk of text once when computing validation loss.
dev_indices = list(range(0, len(dev_dataset), config.block_size))
dev_subset = torch.utils.data.Subset(dev_dataset, dev_indices)
dev_loader = DataLoader(dev_subset, config.batch_size, pin_memory=True)

print( "There are", len(training_dataset), "datapoints and", tokenizer.vocab_size, "token classes in the dataset" ) 

criterion = nn.CrossEntropyLoss()

lm.train()
running_loss = 0
print( datetime.now().strftime("%X"), "Training starts" )
# Can we pick up where we left off?
start_epoch = checkpoint['epoch'] if RESUME_TRAINING else 0
start_iter = checkpoint['iteration'] if RESUME_TRAINING else 0
for epoch in range(start_epoch, config.no_of_epochs) :
    iteration = start_iter if epoch == start_epoch else 0
    # Random sampling of datapoints. Note that we use a new seed for each epoch 
    # -- this will guarantee that we can resume the training anywhere without 
    # repeating any datapoints within an epoch.
    sampler = ResumableRandomSampler(training_dataset, 
                                     start_index=iteration * config.batch_size, 
                                     seed=seed+epoch) 
    training_loader = DataLoader(training_dataset, 
                                 batch_size=config.batch_size, 
                                 sampler=sampler, 
                                 num_workers=4)
    for x,y in training_loader:
        x,y = x.to(device), y.to(device)
        lm_optimizer.zero_grad()
        logits = lm(x)
        loss = criterion(logits.reshape(-1, tokenizer.vocab_size), y.reshape(-1))
        loss.backward()
        lm_optimizer.step()
        running_loss += loss.detach().item()
        iteration += 1
        if iteration%100 == 0:
            # Compute validation loss every 100 iterations
            lm.eval()   
            with torch.no_grad():
                total_loss = 0    
                for vx, vy in dev_loader:
                    vx,vy = vx.to(device), vy.to(device)
                    logits = lm(vx)
                    val_loss = criterion(logits.reshape(-1, tokenizer.vocab_size), vy.reshape(-1))
                    total_loss += val_loss.item()
                print(f'{datetime.now().strftime("%X")} Epoch {epoch+1}, iteration {iteration}: loss={running_loss/100:.4f}, dev loss={total_loss/len(dev_loader):.4f}')
            lm.train()
            running_loss = 0
        if iteration%10000 == 0:
            checkpoint = {
                'model_state_dict': lm.state_dict(),
                'optimizer_state_dict': lm_optimizer.state_dict(),
                'epoch': epoch,
                'iteration': iteration,
                'config': config.__dict__, # Useful so you know the architecture later
            }

            # Overwrites the same file every time to save space
            torch.save(checkpoint, checkpoint_path)
            print(f"Checkpoint saved to {checkpoint_path}")

            # Show progress
            lm.eval()
            with torch.no_grad():
                vx, vy = next(iter(dev_loader))
                vx, vy = vx.to(device), vy.to(device)
                
                sample_x, sample_y = vx[0][:5], vy[0][:5]
                logits = lm(sample_x.unsqueeze(0))
                preds = torch.argmax(logits, dim=-1).squeeze(0)
                
                print(f"\n{'REAL':<15} | {'PREDICCION':<15}")
                print("-" * 33)
                for i in range(len(sample_x)):
                    actual = tokenizer.vocab[sample_y[i].item()]
                    pred = tokenizer.vocab[preds[i].item()]
                    print(f"{actual:<15} | {pred:<15}")
                print("-" * 33 + "\n")
            
            lm.train()

Running on cuda
There are 18507842 datapoints and 5000 token classes in the dataset
14:56:38 Training starts
14:56:45 Epoch 1, iteration 100: loss=5.2421, dev loss=4.4972
14:56:51 Epoch 1, iteration 200: loss=4.3670, dev loss=4.1786
14:56:56 Epoch 1, iteration 300: loss=4.1435, dev loss=4.0246
14:57:02 Epoch 1, iteration 400: loss=4.0182, dev loss=3.9127
14:57:07 Epoch 1, iteration 500: loss=3.8998, dev loss=3.7801
